# Exp 024 — Qwen 3B + long responses (Blind-A)

**Course correction after 023 regressed.** BGE-reranker hurts because Blind-A's ground truth rewards user preference, not semantic similarity — dropping the reranker.

**Two response-side changes vs the 021 baseline (composite 0.33, LLM 3.15):**
1. Qwen 2.5-1.5B → **Qwen 2.5-3B** (more coherent, better metadata citation)
2. `response_max_new_tokens` 64 → **192** (lift the length cap)

**Retrieval unchanged** (wRRF only, no reranker). nDCG@20 should return to ~0.19.

**Target**: composite 0.33 → ~0.36–0.38; LLM 3.15 → ~3.3–3.5; LexDiv 0.67 → ~0.72 (longer responses, more variety).

Wall time: ~5–7 min on A100 (Qwen 3B ~2× slower than 1.5B on 80 rows + longer generation).

Output → `/content/prediction.zip` (CodaBench-compliant) + Drive copy tagged with TID.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone fresh-model.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial

print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%nauthor:  %an%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 3) Install pinned deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters (baked for exp 024).
TID = '024-qwen3b-longresp-blindsetA'
BATCH_SIZE = 16  # Qwen 3B + max_new=192 = larger KV cache; conservative on A100.
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run Blind-A inference: Qwen 3B + max_new_tokens=192, wRRF retrieval unchanged.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate + package CodaBench-compliant prediction.zip.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC), f'inference output missing at {SRC}'
with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 80)')
assert len(rows) == 80
sample = rows[0]
required = {'session_id','user_id','turn_number','predicted_track_ids','predicted_response'}
assert not (required - set(sample.keys()))
assert len(sample['predicted_track_ids']) == 20
assert sample['predicted_response'].strip()

word_lens = sorted(len(r['predicted_response'].split()) for r in rows)
print(f'response words: p25={word_lens[20]} median={word_lens[40]} p75={word_lens[60]} max={word_lens[-1]}')
print('(021 baseline at max_new=64: median 40 words. Expect ~70-90 words now with max_new=192.)')
print(f'sample response[0]: {sample["predicted_response"][:300]!r}')

stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip
print('\nprediction.zip ready — CodaBench-compliant.')

In [ ]:
# 7a) Browser download.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 7b) Drive backup.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/prediction.zip', f'{dst}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst)
!ls -lh {dst}

## Upload `prediction.zip` to CodaBench.

Post-score decomposition will tell us:
- **nDCG@20 should ≈ 0.19** (wRRF restored). If it's != 0.19, there's noise in blind eval we need to account for.
- **LLM Δ attributable** to (Qwen 3B × longer responses). If LLM lifts to ≥ 3.3: validated bigger LM + more tokens is a real win.
- **If composite ≥ 0.37**: best ship yet; ladder up to exp 025 (add fine-tuned reranker, or cf-bpr retrieval).
- **If composite < 0.33**: the Qwen-1.5B → 3B + length delta didn't work either; rethink LM strategy.